|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 6:</h2>|<h1>The Server<h1>|
|<h2>Section:</h2>|<h1>The async engine<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: build the engine loop<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import asyncio
import time

STEP_S = 0.010
print('ready')

Build the engine loop.

One coroutine steps the model forever. Requests arrive on a queue and tokens
leave on per-request queues. Nothing blocks in either direction.

This is stage 15 with the model replaced by `asyncio.sleep`, which is the
right way to get the concurrency right before the GPU is involved.

# Exercise 1: submit, step, stream

Three pieces: a way in, a loop, and a way out. The loop is the only thing
that touches the model.

In [ ]:
class Engine:
  def __init__(self, max_running=8):
    self.waiting     = asyncio.Queue()
    self.running     = {}          # rid -> [remaining, out_queue]
    self.max_running = max_running
    self.stop        = False

  async def submit(self, rid, n_tokens):
    q = asyncio.Queue()
    await self.waiting.put((rid, n_tokens, q))
    return q

  def _admit(self):
    while len(self.running) < self.max_running and not self.waiting.empty():
      rid, n, q = self.waiting.get_nowait()
      self.running[rid] = [n, q]

  async def loop(self):
    while not self.stop:
      self._admit()
      if not self.running:
        await asyncio.sleep(STEP_S); continue
      await asyncio.sleep(STEP_S)              # ONE step, everybody
      for rid, (left, q) in list(self.running.items()):
        self.running[rid][0] -= 1
        q.put_nowait('tok')
        if self.running[rid][0] == 0:
          q.put_nowait(None)                   # end of stream
          del self.running[rid]

  def cancel(self, rid):
    """A client hung up. Free the slot NOW."""
    self.running.pop(rid, None)

eng = Engine(max_running=4)
task = asyncio.create_task(eng.loop())

async def client(rid, n):
  t0 = time.perf_counter(); q = await eng.submit(rid, n); first = None
  while True:
    tok = await q.get()
    if tok is None: break
    if first is None: first = time.perf_counter() - t0
  return first, time.perf_counter() - t0

res = await asyncio.gather(*[client(i, 10) for i in range(8)])
eng.stop = True; await asyncio.sleep(0.05); task.cancel()
print(f'{len(res)} clients done')
print(f'TTFT  min {min(a for a,_ in res):.3f}s  max {max(a for a,_ in res):.3f}s')

# Exercise 2: somebody closes the tab

Watch the number of occupied slots over time. Two clients hang up after five
tokens; their slots should come back immediately, not when the request would
have finished.

In [ ]:
eng = Engine(max_running=4)
task = asyncio.create_task(eng.loop())
occupied = []

async def watcher():
  for _ in range(40):
    occupied.append(len(eng.running))
    await asyncio.sleep(STEP_S)

async def quitter(rid, n, after):
  q = await eng.submit(rid, n)
  for k in range(after):
    if await q.get() is None: return
  eng.cancel(rid)                     # the tab closes

async def stayer(rid, n):
  q = await eng.submit(rid, n)
  while await q.get() is not None: pass

await asyncio.gather(watcher(),
                     *[quitter(i, 30, 5) for i in (0,1)],
                     *[stayer(i, 30) for i in (2,3)])
eng.stop = True; await asyncio.sleep(0.05); task.cancel()

print('slots in use over time:', occupied[:20])
print(f'\npeak {max(occupied)}, and it drops at step ~5 when two clients hang up')

# Exercise 3: against the obvious design

Now the version where the model runs inside the event loop. Measure time to
first token, not throughput.

In [ ]:
async def measure(design, n_clients=8, tokens=15):
  T0 = time.perf_counter(); lat = []
  if design == 'blocking':
    async def c(i):
      f = None
      for _ in range(tokens):
        time.sleep(STEP_S)
        if f is None: f = time.perf_counter()-T0
      lat.append(f)
    await asyncio.gather(*[c(i) for i in range(n_clients)])
  else:
    e = Engine(max_running=n_clients)
    t = asyncio.create_task(e.loop())
    async def c(i):
      q = await e.submit(i, tokens); f = None
      while True:
        tok = await q.get()
        if tok is None: break
        if f is None: f = time.perf_counter()-T0
      lat.append(f)
    await asyncio.gather(*[c(i) for i in range(n_clients)])
    e.stop = True; await asyncio.sleep(0.05); t.cancel()
  return sorted(lat), time.perf_counter()-T0

for design in ('blocking', 'engine'):
  lat, wall = await measure(design)
  print(f'{design:>9}: wall {wall:5.2f}s  TTFT p50 {lat[len(lat)//2]:.3f}s  p99 {lat[-1]:.3f}s')

### What makes this an engine rather than a handler

**One place touches the model.** Not one place per request. That is the
only arrangement in which the in-flight requests are all visible at the
same moment, which is the precondition for batching them, which is the
precondition for everything in Parts 1 and 2.

**Submission returns immediately.** `submit` puts a request on a queue and
hands back another queue. HTTP never waits for the GPU and the GPU never
waits for HTTP.

**Cancellation frees the slot at once.** Without it the sequence keeps
generating into a queue nobody is reading, holding KV blocks until it
reaches its length limit. On real traffic that is not an edge case: it is
what a browser does on every navigation.

**`None` is the end of stream.** A sentinel on the queue, so the client
knows the difference between "no token yet" and "no more tokens".

Real vLLM pushes this further and puts the engine loop in a **separate
process**, because Python on the API side was measurably stalling the
GPU even when it was not blocking. Same principle, one level up.

    ./vc guide 15